In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=10000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([8000, 10]) torch.Size([8000, 1])
torch.Size([2000, 10]) torch.Size([2000, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.5158880352973938
epoch:  1, loss: 0.49586984515190125
epoch:  2, loss: 0.47671186923980713
epoch:  3, loss: 0.45836561918258667
epoch:  4, loss: 0.4407879114151001
epoch:  5, loss: 0.42393991351127625
epoch:  6, loss: 0.4077860713005066
epoch:  7, loss: 0.39229458570480347
epoch:  8, loss: 0.3774356245994568
epoch:  9, loss: 0.363181471824646
epoch:  10, loss: 0.3495055139064789
epoch:  11, loss: 0.33638304471969604
epoch:  12, loss: 0.32379043102264404
epoch:  13, loss: 0.3117053806781769
epoch:  14, loss: 0.30010706186294556
epoch:  15, loss: 0.2889748215675354
epoch:  16, loss: 0.2782897353172302
epoch:  17, loss: 0.26803362369537354
epoch:  18, loss: 0.2581888735294342
epoch:  19, loss: 0.24873925745487213
epoch:  20, loss: 0.23966893553733826
epoch:  21, loss: 0.23096325993537903
epoch:  22, loss: 0.2226075977087021
epoch:  23, loss: 0.21458882093429565
epoch:  24, loss: 0.20689448714256287
epoch:  25, loss: 0.19951443374156952
epoch:  26, loss: 0.19243890047073

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -1840.664671530077
Test metrics:  R2 = -1824.698122339465


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.4842733144760132
epoch:  1, loss: 0.42184990644454956
epoch:  2, loss: 0.35795268416404724
epoch:  3, loss: 0.28672316670417786
epoch:  4, loss: 0.20601965487003326
epoch:  5, loss: 0.1192471832036972
epoch:  6, loss: 0.04980945587158203
epoch:  7, loss: 0.04062409698963165
epoch:  8, loss: 0.03720606118440628
epoch:  9, loss: 0.03214564174413681
epoch:  10, loss: 0.03182557225227356
epoch:  11, loss: 0.03019876591861248
epoch:  12, loss: 0.02905341237783432
epoch:  13, loss: 0.02800072729587555
epoch:  14, loss: 0.026898594573140144
epoch:  15, loss: 0.025840498507022858
epoch:  16, loss: 0.02467932365834713
epoch:  17, loss: 0.023286327719688416
epoch:  18, loss: 0.021713022142648697
epoch:  19, loss: 0.019945034757256508
epoch:  20, loss: 0.018086135387420654
epoch:  21, loss: 0.016233500093221664
epoch:  22, loss: 0.014425255358219147
epoch:  23, loss: 0.012825669720768929
epoch:  24, loss: 0.011455847881734371
epoch:  25, loss: 0.010400302708148956
epoch:  26, l

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.995467835126509
Test metrics:  R2 = 0.994964501331651
